In [4]:
import os
import mlx.core as mx
print(f"Device: {mx.default_device()}")
print(f"HF_TOKEN: {'Set' if os.environ.get('HF_TOKEN') else 'Not Set'}")

Device: Device(gpu, 0)
HF_TOKEN: Set


In [5]:
import os
from mlx_lm import load

# ストレージ管理を徹底（メモの構成に準拠）
os.environ["HF_HOME"] = os.path.expanduser("~/dev/struct-eval-comp/models")

model_id = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Checking/Downloading {model_id} to project folder...")
print(f"Path: {os.environ['HF_HOME']}")

# 8.5GBのDLが始まるので、Wi-Fiの速度を確認してください！
model, tokenizer = load(model_id)

print("\n🎉 ロード成功！struct-eval-comp の準備が完全に整いました。")

Checking/Downloading Qwen/Qwen3-4B-Instruct-2507 to project folder...
Path: /Users/yutako/dev/struct-eval-comp/models


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]


🎉 ロード成功！struct-eval-comp の準備が完全に整いました。


## ダミーデータでテスト

In [5]:
import pandas as pd
import jsonlines

# ダミーデータでのテスト
sample_data = [
    {"input": "この商品の価格を教えて", "output": "{\"price\": 1000}"},
    {"input": "在庫はありますか？", "output": "{\"stock\": \"yes\"}"}
]

def augment_data(data):
    augmented = []
    for item in data:
        augmented.append(item) # オリジナル
        # 例：言い換えやプロンプトのバリエーションを増やすロジック
        new_item = item.copy()
        new_item["input"] = "質問です。" + item["input"] 
        augmented.append(new_item)
    return augmented

print(f"拡張前: {len(sample_data)} -> 拡張後: {len(augment_data(sample_data))}")

拡張前: 2 -> 拡張後: 4


In [6]:
import json

# --- ステップ1: 元データ（明日の配布データだと思ってください） ---
original_data = [
    {"input": "価格を教えて", "output": "1000円"},
    {"input": "在庫は？", "output": "あり"}
]

# --- ステップ2: 自分でルールを決めて増やす（拡張） ---
def my_augment(data):
    new_dataset = []
    for item in data:
        # (A) そのまま入れる
        new_dataset.append(item)
        
        # (B) 自分で考えた言い換えパターンを追加
        # 例：語尾を丁寧にしてみる、など
        variant = item.copy()
        variant["input"] = item["input"] + "ください。" 
        new_dataset.append(variant)
        
    return new_dataset

# --- ステップ3: 結果の確認（画面に出して見る） ---
augmented = my_augment(original_data)

print(f"元の数: {len(original_data)} -> 拡張後: {len(augmented)}\n")
for i, d in enumerate(augmented):
    print(f"{i}: {d['input']} -> {d['output']}")

元の数: 2 -> 拡張後: 4

0: 価格を教えて -> 1000円
1: 価格を教えてください。 -> 1000円
2: 在庫は？ -> あり
3: 在庫は？ください。 -> あり


## 軽い拡張

In [ ]:
import random

def light_augment(text):
    # ネット不要！単純なルールでバリエーションを作る
    variants = [text]
    # 1. 最後にピリオドがないなら足してみる
    if not text.endswith("."):
        variants.append(text + ".")
    # 2. 大文字・小文字を入れ替えてみる（英語データ想定）
    variants.append(text.lower())
    variants.append(text.upper())
    return list(set(variants)) # 重複削除

## BERTの拡張準備

In [8]:
import os
from mlx_lm import load # Qwen用
import nlpaug.augmenter.word as naw # BERT用

# 鉄則のパス固定
os.environ["HF_HOME"] = os.path.expanduser("~/dev/struct-eval-comp/models")

# BERTのダウンロード（約500MB〜）
# multilingual-casedなら日本語の単語置換もいけます
print("BERTモデルをロード中...")
# aug = naw.ContextualWordEmbsAug(
#     model_path='bert-base-multilingual-cased',
#     action="substitute",
#     device='cpu'
# )
print("BERT準備完了！")

BERTモデルをロード中...
BERT準備完了！


### BERT拡張のテスト

In [1]:
import os
import torch  # ← これが抜けていました！
from transformers import pipeline

# 鉄則のパス固定
os.environ["HF_HOME"] = os.path.expanduser("~/dev/struct-eval-comp/models")

model_id = 'bert-base-multilingual-cased'

print(f"BERT ({model_id}) をロード中...")

# deviceの設定を安全に行う
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 穴埋めパイプラインの作成
fill_mask = pipeline(
    "fill-mask", 
    model=model_id, 
    device=device
)

# テスト用の文章（BERTは [MASK] を埋めるのが仕事です）
text = "この [MASK] は非常に重要です。"

print("\n--- 置換テスト ---")
results = fill_mask(text)

for i, res in enumerate(results[:3]):
    print(f"候補 {i+1}: {res['sequence']}")

print("\n🎉 BERTのロードも完了！これで全ての武器が揃いました。")

BERT (bert-base-multilingual-cased) をロード中...


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- 置換テスト ---
候補 1: この 章 は 非 常 に 重 要 です 。
候補 2: このとき は 非 常 に 重 要 です 。
候補 3: この 事 は 非 常 に 重 要 です 。

🎉 BERTのロードも完了！これで全ての武器が揃いました。


## 評価データの確認

In [6]:
import json
from pathlib import Path

data_path = Path("/Users/yutako/dev/struct-eval-comp/data/public_150.json").expanduser()

def analyze_samples(file_path, count=10):
    samples = []
    with open(file_path, 'r', encoding='utf-8') as f:
        # for i, line in enumerate(f):
        #     if i >= count: break
            # samples.append(json.loads(line))
            samples = json.load(f)
    
    # 1. データの「キー」を網羅
    all_keys = set().union(*(s.keys() for s in samples))
    print(f"🔑 全体で使われているキー: {all_keys}")

    # 2. テキストの長さの統計
    lengths = [len(str(s.get('instruction', ''))) + len(str(s.get('input', ''))) for s in samples]
    print(f"📏 テキスト長: 最短={min(lengths)}, 最長={max(lengths)}, 平均={sum(lengths)/len(lengths):.1f}")

    # 3. 最初の3件だけ詳細表示
    for i in range(min(10, len(samples))):
        print(f"\n--- Sample {i} ---")
        print(json.dumps(samples[i], indent=2, ensure_ascii=False))

# 実行（今夜データを入れたら！）
analyze_samples(data_path)

🔑 全体で使われているキー: {'rendering', 'task_name', 'output_type', 'task_id', 'query'}
📏 テキスト長: 最短=0, 最長=0, 平均=0.0

--- Sample 0 ---
{
  "task_id": "p_7b3394e21698627665533715",
  "task_name": "Text to JSON",
  "rendering": false,
  "query": "Please output JSON code:\n\nTask:\nSummarize a fictional ecosystem with detailed information about its climate, species, and threats.\n\nFeature Requirements:\n1. The field name is 'name', which is a string representing the official designation of the ecosystem, located directly within the 'ecosystem' object.\n2. The key 'latitude' is a number indicating the geographic latitude of the ecosystem's location, nested inside 'location', which itself is within 'ecosystem'.\n3. The key 'longitude' is a number specifying the geographic longitude of the ecosystem's location, found within 'location' under 'ecosystem'.\n4. The field 'type' is a string describing the general climate category (such as tropical, arid, or temperate), located in the 'climate' object inside

## テストトレーニング（２月3日０４：４０）の結果確認

In [7]:
from mlx_lm import load, generate

model_id = "Qwen/Qwen3-4B-Instruct-2507"
adapter_path = "test_adapter"

# 1. テスト用の「ちょっと意地悪な問題」を用意
# （さっき学習した Hard データセットに近い形式です）
test_prompt = """Convert the following text into a JSON object.
Text: "Ichigo loves strawberries. He was born on July 7th."
Constraint: Use keys 'name', 'hobby', and 'birthday'.
Return ONLY JSON."""

# apply_chat_template 形式に整形（ここが大事！）
messages = [{"role": "user", "content": test_prompt}]
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("🚀 【実験1】素のQwen3-4B の回答:")
model_base, tokenizer_base = load(model_id)
response_base = generate(model_base, tokenizer_base, prompt=prompt, max_tokens=100)
print(response_base)

print("\n" + "="*50 + "\n")

print("🔥 【実験2】特訓後（アダプタあり）の回答:")
# アダプタをセットしてロード
model_sft, tokenizer_sft = load(model_id, adapter_path=adapter_path)
response_sft = generate(model_sft, tokenizer_sft, prompt=prompt, max_tokens=100)
print(response_sft)

🚀 【実験1】素のQwen3-4B の回答:


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.


{"name": "Ichigo", "hobby": "strawberries", "birthday": "July 7th"}


🔥 【実験2】特訓後（アダプタあり）の回答:


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

{"name": "Ichigo", "hobby": "strawberries", "birthday": "July 7th"}


## 提出スクリプト

In [1]:
import json
import os
from mlx_lm import load, generate
from tqdm import tqdm

# 設定
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTER_PATH = "test_adapter"
INPUT_FILE = "data/public_150.json"
OUTPUT_FILE = "submissions/submission.json"

os.makedirs("submissions", exist_ok=True)

print(f"📦 モデルとアダプタをロード中...")
model, tokenizer = load(MODEL_ID, adapter_path=ADAPTER_PATH)

print(f"📖 本番データを読み込み中: {INPUT_FILE}")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

results = []

print(f"🔥 推論開始（150問）...")
for item in tqdm(data):
    # プロンプトの作成（学習時と同じ形式にするのがコツです）
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": item["query"]}],
        tokenize=False,
        add_generation_prompt=True
    )
    
    # 推論実行（M4ならここが速い！）
    response = generate(model, tokenizer, prompt=prompt, max_tokens=2048)
    
    # 結果を保存
    results.append({
        "task_id": item["task_id"],
        "answer": response.strip()
    })

# JSON形式で書き出し
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\n✅ 完了！ 提出用ファイルが作成されました: {OUTPUT_FILE}")

📦 モデルとアダプタをロード中...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

📖 本番データを読み込み中: data/public_150.json
🔥 推論開始（150問）...


  5%|▍         | 7/150 [04:31<1:32:20, 38.75s/it]


KeyboardInterrupt: 

In [2]:
# 仮想環境の python で実行
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="./test_adapter",
    repo_id="satoyutaka/llm2026_main",
    repo_type="model"
)

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-698113af-2b03274673d0297f340e72d2;b086b0f8-01b6-425e-a694-1ad971efb974)

Repository Not Found for url: https://huggingface.co/api/models/satoyutaka/llm2026_main/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [5]:
from huggingface_hub import HfApi, create_repo
import os

# 設定
repo_id = "satoyutaka/llm2026_main"
folder_path = "./test_adapter"
# トークンを明示的に使用
token = os.environ.get("HF_TOKEN")

api = HfApi(token=token)

print(f"🚀 リポジトリの確認/作成中: {repo_id}")
try:
    # private=True にすると他の人に見られず安全です。
    # 公開して良ければ False にしてください。
    create_repo(repo_id=repo_id, token=token, private=True, exist_ok=True)
    print("✅ リポジトリ準備完了")
except Exception as e:
    print(f"リポジトリ作成時に通知（無視してOKな場合が多い）: {e}")

print(f"📦 アップロードを開始します...")
try:
    api.upload_folder(
        folder_path=folder_path,
        repo_id=repo_id,
        repo_type="model",
        token=token
    )
    print(f"✅✅ 成功！！\nURL: https://huggingface.co/{repo_id}")
except Exception as e:
    print(f"❌ アップロード失敗: {e}")

🚀 リポジトリの確認/作成中: satoyutaka/llm2026_main
リポジトリ作成時に通知（無視してOKな場合が多い）: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-698114d1-50fb5ce05e24d1d11d1fd01c;55c3f860-773b-43ed-ae16-0fe6be144f20)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.
📦 アップロードを開始します...
❌ アップロード失敗: 401 Client Error. (Request ID: Root=1-698114d1-4745235a6e4a1679590adf20;10b7c912-665d-430a-ad61-d7b09e2bb27d)

Repository Not Found for url: https://huggingface.co/api/models/satoyutaka/llm2026_main/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the c

In [6]:
from huggingface_hub import HfApi
import os

# 設定
repo_id = "satoyutaka/qwen3-merged-test" # 新しいリポ名
folder_path = "./fused_model_qwen"      # さっき作った合体モデル
token = os.environ.get("HF_TOKEN")

api = HfApi(token=token)

print(f"🚀 合体モデルのアップロードを開始します: {repo_id}")
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

api.upload_folder(
    folder_path=folder_path,
    repo_id=repo_id,
    repo_type="model",
    token=token
)
print(f"✅✅ 成功！！\nURL: https://huggingface.co/{repo_id}")

🚀 合体モデルのアップロードを開始します: satoyutaka/qwen3-merged-test


HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-69811bae-282c246d0671b907010fca3e;966d4e57-dfec-4120-92c3-2a6979395ab6)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# 環境設定
os.environ["HF_HOME"] = os.path.expanduser("~/dev/struct-eval-comp/models")
model_id = "Qwen/Qwen3-4B-Instruct-2507"
data_path = os.path.expanduser("~/dev/struct-eval-comp/data/train_data/train.jsonl")

# 1. ロード
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.bfloat16, 
    device_map={"": "mps"}
)

# 2. データ準備（10件テストから全件へ切り替え可能）
dataset = load_dataset("json", data_files=data_path, split="train")
def format_func(sample):
    # JSONLのキー名に合わせてここを修正（例: instruction, output）
    return {"text": f"### 指示\n{sample.get('instruction', '')}\n\n### 応答\n{sample.get('output', '')}"}
dataset = dataset.map(format_func)

# 3. LoRA設定（これが標準形式のアダプタになる）
peft_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"
)

# 4. 学習設定（夜は max_steps を増やす）
training_args = SFTConfig(
    output_dir="./output_standard_lora",
    max_steps=100, # 本番は 500〜1000 程度
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # メモリ節約しつつバッチサイズを稼ぐ
    learning_rate=2e-4,
    dataset_text_field="text",
    save_strategy="no"
)

# 5. 実行 & 保存
trainer = SFTTrainer(model=model, args=training_args, train_dataset=dataset, peft_config=peft_config)
trainer.train()
trainer.model.save_pretrained("./final_standard_adapter") # これが宝物（数百MB）

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/yutako/dev/struct-eval-comp/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.376991
20,1.062639
30,0.859123
40,0.745668
50,0.684340
60,0.648946
70,0.629514
80,0.619687
90,0.628728
100,0.628782


In [ ]:
!mkdir -p data/train_data1 && head -n 100 data/hf_datasets/structured-hard-sft-4k.jsonl > data/train_data1/train1.jsonl && tail -n 10 data/hf_datasets/structured-hard-sft-4k.jsonl > data/train_data1/valid1.jsonl

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# 環境設定
os.environ["HF_HOME"] = os.path.expanduser("~/dev/struct-eval-comp/models")
model_id = "Qwen/Qwen3-4B-Instruct-2507"
data_path = os.path.expanduser("~/dev/struct-eval-comp/data/train_data/train.jsonl")

# 1. ロード
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.bfloat16, 
    device_map={"": "mps"}
)

# 2. データ準備（10件テストから全件へ切り替え可能）
dataset = load_dataset("json", data_files=data_path, split="train")
def format_func(sample):
    # JSONLのキー名に合わせてここを修正（例: instruction, output）
    return {"text": f"### 指示\n{sample.get('instruction', '')}\n\n### 応答\n{sample.get('output', '')}"}
dataset = dataset.map(format_func)

# 3. LoRA設定（これが標準形式のアダプタになる）
peft_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"
)

# 4. 学習設定（夜は max_steps を増やす）
training_args = SFTConfig(
    output_dir="./output_standard_lora",
    max_steps=100, # 本番は 500〜1000 程度
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # メモリ節約しつつバッチサイズを稼ぐ
    learning_rate=2e-4,
    dataset_text_field="text",
    save_strategy="no"
)

# 5. 実行 & 保存
trainer = SFTTrainer(model=model, args=training_args, train_dataset=dataset, peft_config=peft_config)
trainer.train()
trainer.model.save_pretrained("./final_standard_adapter") # これが宝物（数百MB）